Cell 0

cell 1

In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
!pip install datasets av  # uncomment on first run

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 24.7 MB/s  0:00:01


cell 2

In [2]:
import sys
import random
import datetime
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "fix/teacher-full-16frame-clips"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Cloned → {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

Cloned → /content/anyup
PyTorch : 2.14.1+cu130
CUDA    : True
GPU     : Tesla T4


cell 3

In [3]:
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class TStage:
    t:           int
    start_step:  int
    batch_size:  int


@dataclass
class TrainConfig:
    # ── Experiment ────────────────────
    experiment_name: str = "run_02"

    # ── Paths ───────────────────────
    model_ckpt_2d:  Optional[str] = "checkpoints/anyup_2d.pth"
    checkpoint_dir: str           = "checkpoints/anyup3d"
    resume:         Optional[str] = None
    log_dir:        str           = "runs/anyup3d"
    dataset_root:   str           = "data/videos"

    # ── Teacher ───────────────────────
    encoder:       str = "videomae"
    encoder_model: str = "MCG-NJU/videomae-base"

    # ── Model ───────────────────────
    input_dim:       int   = 3
    qk_dim:          int   = 128    # ↓ reduce to 64 to save memory
    num_heads:       int   = 4      # must divide qk_dim evenly
    window_ratio:    float = 0.1    # ↓ reduce to cut attention cost; also affects memory
    kernel_size_lfu: int   = 5
    t_k:             int   = 3
    window_t:         int   = 3


    # ── Spatial resolution ───────────────────────
    # Relationship: img_size = crop_size // 2, source_size > crop_size
    source_size: int = 256   # frames loaded at this size; must be > crop_size
    crop_size:   int = 224   # spatial tube crop fed to VideoMAE for GT (teacher native)
                             #   ↑ if you change this, also update img_size below
    img_size:    int = 112   # AnyUp input (= crop_size // 2)
                             #   ↑ must be crop_size // 2 for 2× upsample to work

    # ── T-curriculum ───────────────────────
    t_schedule: List[TStage] = field(default_factory=lambda: [
        TStage(t=2,  start_step=0,     batch_size=2),   # ↓ reduce batch_size if OOM
        TStage(t=4,  start_step=5000,  batch_size=1),   # depends on t=2 batch above
        TStage(t=8,  start_step=15000, batch_size=1),   # depends on t=4 batch above
        # TStage(t=16, start_step=30000, batch_size=1), # uncomment for A100 only
    ])
    frame_stride: int = 4   # teacher clip stride (VideoMAE pretrain = 4); 16 frames span 61 source frames
                            #   ↑ used by UCF101TopNDataset._sample_frames (cell 9); raises decode cost per clip

    # ── Optimizer ───────────────────────
    lr:                 float = 2e-4
    weight_decay:       float = 1e-4
    grad_clip_max_norm: float = 1.0

    # ── Loss weights ───────────────────────
    lambda_cos_mse:               float = 1.0
    lambda_input_consistency:     float = 0.1
    lambda_self_consistency:      float = 0.1
    lambda_temporal_consistency:  float = 0.2
    temporal_lambda_warmup_steps: int   = 5000

    # ── Training duration ───────────────────────
    max_steps:          int = 50_000
    save_every_n_steps: int = 250
    val_every_n_steps:  int = 1000
    log_every_n_steps:  int = 50

    # ── Data ───────────────────────
    num_workers: int  = 2    # set to 0 if Colab throws Bus errors
    pin_memory:  bool = True

    # ── Misc ───────────────────────
    seed:        int  = 42
    debug:       bool = False
    debug_steps: int  = 10


cfg = TrainConfig()

# ============================================================
# COLAB OVERRIDES — edit this block for every new run.
# ============================================================
DRIVE_ROOT = "/content/drive/MyDrive/anyup3d"   # ← your Drive folder

cfg.experiment_name = "run_03"
cfg.model_ckpt_2d   = f"{DRIVE_ROOT}/anyup_2d.pth"
cfg.checkpoint_dir  = f"{DRIVE_ROOT}/checkpoints/{cfg.experiment_name}"
cfg.log_dir         = f"{DRIVE_ROOT}/runs/{cfg.experiment_name}"
cfg.dataset_root    = "/content/local_dataset"

cfg.resume = None   # run_02 checkpoints were trained on degraded (short-clip) teacher GT — retrain from 2D init
# cfg.debug  = True

print(cfg)

TrainConfig(experiment_name='run_03', model_ckpt_2d='/content/drive/MyDrive/anyup3d/anyup_2d.pth', checkpoint_dir='/content/drive/MyDrive/anyup3d/checkpoints/run_03', resume=None, log_dir='/content/drive/MyDrive/anyup3d/runs/run_03', dataset_root='/content/local_dataset', encoder='videomae', encoder_model='MCG-NJU/videomae-base', input_dim=3, qk_dim=128, num_heads=4, window_ratio=0.1, kernel_size_lfu=5, t_k=3, window_t=3, source_size=256, crop_size=224, img_size=112, t_schedule=[TStage(t=2, start_step=0, batch_size=2), TStage(t=4, start_step=5000, batch_size=1), TStage(t=8, start_step=15000, batch_size=1)], frame_stride=4, lr=0.0002, weight_decay=0.0001, grad_clip_max_norm=1.0, lambda_cos_mse=1.0, lambda_input_consistency=0.1, lambda_self_consistency=0.1, lambda_temporal_consistency=0.2, temporal_lambda_warmup_steps=5000, max_steps=50000, save_every_n_steps=250, val_every_n_steps=1000, log_every_n_steps=50, num_workers=2, pin_memory=True, seed=42, debug=False, debug_steps=10)


cell 4

In [4]:
import numpy as np

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


cell 5

In [5]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")
except ModuleNotFoundError:
    print("Not in Colab — Drive mount skipped.")

timestamp   = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_log_dir = Path(cfg.log_dir) / timestamp
run_log_dir.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(run_log_dir))

print(f"TensorBoard log dir : {run_log_dir}")
print("Launch with:  %load_ext tensorboard")
print(f"              %tensorboard --logdir {cfg.log_dir}")

Mounted at /content/drive
Drive mounted.
TensorBoard log dir : /content/drive/MyDrive/anyup3d/runs/run_03/2026-11-15_11-09-53
Launch with:  %load_ext tensorboard
              %tensorboard --logdir /content/drive/MyDrive/anyup3d/runs/run_03


cell 6

In [6]:
from transformers import VideoMAEModel

print(f"Loading teacher: {cfg.encoder} — {cfg.encoder_model} ...")

if cfg.encoder == "videomae":
    encoder      = VideoMAEModel.from_pretrained(cfg.encoder_model).to(device).eval()
    ENCODER_DIM  = encoder.config.hidden_size    # 768
    PATCH_SIZE   = encoder.config.patch_size     # 16
    TEACHER_SIZE = encoder.config.image_size     # 224
    TUBELET_SIZE = encoder.config.tubelet_size   # 2
    TEACHER_T    = encoder.config.num_frames     # 16 — every build_loader call and the window sampling depend on this
else:
    raise ValueError(
        f"Unsupported encoder '{cfg.encoder}'. "
        f"Add a loading branch here before using a new encoder."
    )

for p in encoder.parameters():
    p.requires_grad_(False)

TOKEN_H     = cfg.crop_size // PATCH_SIZE    # 14
TOKEN_W     = cfg.crop_size // PATCH_SIZE    # 14
TOKEN_H_LOW = cfg.img_size  // PATCH_SIZE    # 7
TOKEN_W_LOW = cfg.img_size  // PATCH_SIZE    # 7

print(f"ENCODER_DIM : {ENCODER_DIM}")
print(f"GT tokens   : {TOKEN_H}×{TOKEN_W}   (from {cfg.crop_size}-px crop)")
print(f"Coarse tok. : {TOKEN_H_LOW}×{TOKEN_W_LOW}  (from {cfg.img_size}-px input)")

Loading teacher: videomae — MCG-NJU/videomae-base ...


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  377MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.head.bias                                                    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.bias   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.key.weight   | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.bias      | UNEXPECTED |  | 
mask_token                                                           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.atten

ENCODER_DIM : 768
GT tokens   : 14×14   (from 224-px crop)
Coarse tok. : 7×7  (from 112-px input)


Check Cell 6.5

In [7]:
import inspect, torch, transformers
from transformers.models.videomae import modeling_videomae as m

pe = encoder.embeddings.position_embeddings
print("transformers :", transformers.__version__)
print("type         :", type(pe).__name__, "| is Parameter:", isinstance(pe, torch.nn.Parameter))
print("shape        :", tuple(pe.shape))

src = inspect.getsource(m.VideoMAEEmbeddings.forward)
print("forward mentions 'interpolat':", "interpolat" in src)
print(src)

transformers : 5.22.1
type         : Tensor | is Parameter: False
shape        : (1, 1568, 768)
forward mentions 'interpolat': False
    def forward(self, pixel_values, bool_masked_pos):
        # create patch embeddings
        embeddings = self.patch_embeddings(pixel_values)

        # add position embeddings
        embeddings = embeddings + self.position_embeddings.detach().type_as(embeddings).to(
            device=embeddings.device, copy=True
        )
        # only keep visible patches
        # ~bool_masked_pos means visible
        if bool_masked_pos is not None:
            batch_size, _, num_channels = embeddings.shape
            embeddings = embeddings[~bool_masked_pos]
            embeddings = embeddings.reshape(batch_size, -1, num_channels)

        return embeddings



cell 7

In [8]:
from anyup.model import AnyUp
from scripts.load_2d_weights import build_3d_state_dict

anyup = AnyUp(
    input_dim=cfg.input_dim,
    qk_dim=cfg.qk_dim,
    num_heads=cfg.num_heads,
    window_ratio=cfg.window_ratio,
    kernel_size_lfu=cfg.kernel_size_lfu,
    t_k=cfg.t_k,
).to(device)

if cfg.model_ckpt_2d and Path(cfg.model_ckpt_2d).exists():
    print(f"Loading 2D checkpoint: {cfg.model_ckpt_2d}")
    raw = torch.load(cfg.model_ckpt_2d, map_location="cpu", weights_only=False)
    sd  = raw
    for key in ("anyup", "model", "state_dict", "weights", "params"):
        if isinstance(sd, dict) and key in sd:
            sd = sd[key]; break
    adapted_sd, _ = build_3d_state_dict(sd, anyup)
    anyup.load_state_dict(adapted_sd, strict=True)
    print(f"  Loaded {len(adapted_sd)} tensors with temporal centre-init")
elif cfg.model_ckpt_2d:
    print(f"[WARNING] 2D ckpt not found at '{cfg.model_ckpt_2d}' — random init")
else:
    print("Random init (no 2D checkpoint configured)")

global_step = 0
data_epoch  = 0

if cfg.resume and Path(cfg.resume).exists():
    print(f"Resuming from: {cfg.resume}")
    ckpt        = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    anyup.load_state_dict(ckpt["anyup"])
    global_step = ckpt.get("global_step", 0)
    data_epoch  = ckpt.get("data_epoch",  0)
    print(f"  Resumed at step {global_step}, data_epoch {data_epoch}")

scaler = torch.amp.GradScaler('cuda')  # correct — device goes here in the new API
print(f"Trainable params: {sum(p.numel() for p in anyup.parameters() if p.requires_grad):,}")

Loading 2D checkpoint: /content/drive/MyDrive/anyup3d/anyup_2d.pth
  Loaded 73 tensors with temporal centre-init
Trainable params: 1,540,736


cell 8

In [9]:
from anyup.loss import Cosine_MSE

criterion = Cosine_MSE()


def cosine_mse_3d(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    B, C, T, H, W = pred.shape
    return criterion(pred.view(B, C, T * H, W), target.view(B, C, T * H, W))["total"]


def input_consistency_loss(
    pred:     torch.Tensor,
    guidance: torch.Tensor,
) -> torch.Tensor:
    B, C, T, H, W   = pred.shape
    _, _, _, Hg, Wg = guidance.shape

    pred_flat     = pred.permute(0, 2, 1, 3, 4).reshape(B * T, C, H, W)
    guidance_flat = guidance.permute(0, 2, 1, 3, 4).reshape(B * T, C, Hg, Wg)

    pred_pooled = F.adaptive_avg_pool2d(pred_flat, (Hg, Wg))

    pred_coarse = pred_pooled.view(B, T, C, Hg, Wg).permute(0, 2, 1, 3, 4)
    guid_coarse = guidance_flat.view(B, T, C, Hg, Wg).permute(0, 2, 1, 3, 4)

    return cosine_mse_3d(pred_coarse, guid_coarse)


def get_temporal_lambda(step: int) -> float:
    warmup = cfg.temporal_lambda_warmup_steps
    if warmup <= 0:
        return cfg.lambda_temporal_consistency
    return cfg.lambda_temporal_consistency * min(1.0, step / warmup)


print("Loss functions ready.")

Loss functions ready.


cell 9

In [10]:
import os
import random
import torch
import av
import pandas as pd
import torchvision.transforms.v2 as Tv2
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import InterpolationMode
from google.colab import drive
from anyup.data.training.teacher import assemble_clip, sample_clip_indices

# ==========================================
# 1. MOUNT & EXTRACT (DRIVE ONLY)
# ==========================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Paths - Verify these match your Drive exactly
DRIVE_TAR = Path("/content/drive/MyDrive/anyup3d/ucf101_top10.tar.gz")
WORK_ROOT = Path("/content/ucf101_top10")

if not WORK_ROOT.exists():
    if DRIVE_TAR.exists():
        print(f"✅ Archive found. Extracting {DRIVE_TAR} to local storage...")
        # Use !tar for high-speed extraction from Drive mount to local NVMe
        !tar -xzf {DRIVE_TAR} -C /content/
        print("Extraction complete.")
    else:
        raise FileNotFoundError(f"❌ Could not find {DRIVE_TAR}. Check your folder name/path.")
else:
    print("🚀 Dataset already exists in local storage. Skipping extraction.")

# ==========================================
# 2. GLOBAL CONFIGURATION
# ==========================================

TRAIN_DIR = WORK_ROOT / "train"
TEST_DIR  = WORK_ROOT / "test"
TRAIN_CSV = WORK_ROOT / "train.csv"
TEST_CSV  = WORK_ROOT / "test.csv"

# ==========================================
# 3. TRANSFORMS & DATASET CLASS
# ==========================================
frame_tf = Tv2.Compose([
    Tv2.ToImage(),
    Tv2.Resize((cfg.source_size, cfg.source_size),
               interpolation=InterpolationMode.BILINEAR, antialias=True),
    Tv2.CenterCrop(cfg.source_size),
    Tv2.ToDtype(torch.float32, scale=True),
    Tv2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class UCF101TopNDataset(Dataset):
    def __init__(self, csv_path, video_dir, t=16, stride=4):
        self.df           = pd.read_csv(csv_path).reset_index(drop=True)
        self.video_dir    = Path(video_dir)
        self.t            = t        # frames per clip (TEACHER_T = 16)
        self.stride       = stride   # cfg.frame_stride; stride 4 → decode ~61 frames per clip
        self.classes      = sorted(self.df["tag"].unique().tolist())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def _sample_frames(self, path):
        try:
            container = av.open(str(path))
            stream    = container.streams.video[0]
            total     = stream.frames or self.t * self.stride

            # Temporal sampling: self.t frames at self.stride (smaller stride on short clips)
            idxs      = sample_clip_indices(total, self.t, self.stride)
            wanted    = set(idxs)
            got       = {}

            for fi, frm in enumerate(container.decode(video=0)):
                if fi in wanted:
                    got[fi] = frame_tf(frm.to_image().convert("RGB"))
                if fi >= idxs[-1]:
                    break
            container.close()
            frames = assemble_clip(got, idxs)   # skips undecoded indices, repeats the last frame
        except Exception:
            frames = []

        # Padding for short/broken videos
        while len(frames) < self.t:
            frames.append(frames[-1].clone() if frames
                          else torch.zeros(3, cfg.source_size, cfg.source_size))
        return frames[:self.t]

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        frames = self._sample_frames(self.video_dir / row["video_name"])
        return {
            "video": torch.stack(frames), # [T, C, H, W]
            "label": self.class_to_idx[row["tag"]],
            "tag":   row["tag"],
        }

# ==========================================
# 4. DATALOADER BUILDER
# ==========================================
def build_loader(t: int, batch_size: int, epoch: int = 0, split: str = "train") -> DataLoader:
    csv  = TRAIN_CSV if split == "train" else TEST_CSV
    vdir = TRAIN_DIR if split == "train" else TEST_DIR
    ds   = UCF101TopNDataset(csv_path=csv, video_dir=vdir, t=t, stride=cfg.frame_stride)

    g = torch.Generator()
    g.manual_seed(cfg.seed + epoch)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(split == "train"),
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        generator=g,
        drop_last=(split == "train"),
    )

# ==========================================
# 5. QUICK TEST
# ==========================================
try:
    test_loader = build_loader(t=TEACHER_T, batch_size=2, split="train")
    sample = next(iter(test_loader))
    print(f"\n✅ Success! Loaded batch of shape: {sample['video'].shape}")
    print(f"Tags in batch: {sample['tag']}")
except Exception as e:
    print(f"\n❌ Setup failed: {e}")

FileNotFoundError: ❌ Could not find /content/drive/MyDrive/anyup3d/ucf101_top10.tar.gz. Check your folder name/path.

cell 10

In [ ]:
optimizer = torch.optim.AdamW(anyup.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.max_steps, eta_min=cfg.lr * 0.01)

if cfg.resume and Path(cfg.resume).exists():
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    if "optimizer" in ckpt: optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt: scheduler.load_state_dict(ckpt["scheduler"])

print("Optimiser and scheduler ready.")

cell 11

In [ ]:
ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)


def save_checkpoint(step: int, data_epoch: int, tag: str = "") -> Path:
    name = f"anyup3d_step{step}{('_' + tag) if tag else ''}.pth"
    path = ckpt_dir / name
    torch.save({
        "anyup":       anyup.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "scheduler":   scheduler.state_dict(),
        "global_step": step,
        "data_epoch":  data_epoch,
        "cfg":         vars(cfg),
    }, path)
    print(f"Checkpoint → {path}")
    return path


print(f"Checkpoints → {ckpt_dir}")

cell 12

In [ ]:
from anyup.data.training.teacher import extract_teacher_features, sample_student_window, take_window
# Teacher always runs on TEACHER_T = 16 frames with the unmodified position table;
# the student's window is sliced out of that pass (no position-embedding interpolation).


def sample_tube_crop(video: torch.Tensor):
    CH = cfg.crop_size   # 224
    IL = cfg.img_size    # 112

    S_h, S_w = video.shape[3], video.shape[4]   # actual frame dims

    y = random.randint(0, S_h - CH)
    x = random.randint(0, S_w - CH)

    crop_high = video[:, :, :, y : y + CH, x : x + CH].contiguous()

    B_v, T_v = crop_high.shape[:2]
    crop_low  = F.interpolate(
        crop_high.reshape(B_v * T_v, 3, CH, CH),
        size=(IL, IL),
        mode="bilinear", align_corners=False,
    ).view(B_v, T_v, 3, IL, IL)

    return crop_high, crop_low


@torch.no_grad()
def get_rgb_diff_gate(crop_low: torch.Tensor, T_tok: int) -> torch.Tensor:
    B, T, C, H, W = crop_low.shape

    token_frames = crop_low[:, ::TUBELET_SIZE]
    frame_diff   = (token_frames[:, 1:] - token_frames[:, :-1]).abs()
    frame_diff   = frame_diff.mean(dim=2, keepdim=True)

    Tt   = T_tok - 1
    gate = F.interpolate(
        frame_diff.reshape(B * Tt, 1, H, W),
        size=(TOKEN_H, TOKEN_W),
        mode="bilinear", align_corners=False,
    ).reshape(B, Tt, 1, TOKEN_H, TOKEN_W)

    gate = gate / (gate.amax(dim=(2, 3, 4), keepdim=True) + 1e-6)
    gate = 1.0 - gate

    return gate.permute(0, 2, 1, 3, 4)


def get_current_stage(step: int) -> TStage:
    stage = cfg.t_schedule[0]
    for s in cfg.t_schedule:
        if step >= s.start_step:
            stage = s
    return stage


print("Training utilities ready.")

Cell 12.5

In [ ]:

# ── Cell 12.5 — TEMPORARY debug-run setup for the teacher fix. Don't save the notebook. ──
# Shortens the run, sends debug output to separate Drive folders, and records time, GPU memory and losses.
# Uses no GPU memory itself: it only reads PyTorch's memory counters and the clock.
import time, math
from collections import defaultdict
from torch.utils.data import DataLoader, Subset

# ── Short run: 5 steps per stage, one (short) validation at the end ──
cfg.debug, cfg.debug_steps = True, 15                    # 15 training steps in total
cfg.val_every_n_steps      = 15                          # one validation pass, after the last step (T=8)
cfg.log_every_n_steps      = 1                           # log every step's losses
cfg.t_schedule = [TStage(t=2, start_step=0,  batch_size=2),   # steps 0–4
                  TStage(t=4, start_step=5,  batch_size=1),   # steps 5–9
                  TStage(t=8, start_step=10, batch_size=1)]   # steps 10–14
VAL_LIMIT = 50                                           # ↓ validation clips timed; the full-split time is estimated

# ── Debug output goes to its own folders, not run_03 ──
ckpt_dir = Path(f"{DRIVE_ROOT}/checkpoints/debug_teacher"); ckpt_dir.mkdir(parents=True, exist_ok=True)

_scalars = defaultdict(list)                             # tag -> [(step, value)]
class _RecWriter(SummaryWriter):                         # still writes TensorBoard logs, and keeps a copy of each value
    def add_scalar(self, tag, value, global_step=None, *a, **k):
        _scalars[tag].append((global_step, float(value)))
        return super().add_scalar(tag, value, global_step, *a, **k)
writer = _RecWriter(log_dir=f"{DRIVE_ROOT}/runs/debug_teacher")

# ── Recorders ──
_stamps = []                                             # (clock time, T) at the start of every step
_peak   = {}                                             # T -> (peak allocated, peak reserved) in bytes
_t      = {}                                             # validation marks

def _unwrap(f):                                          # safe to re-run this cell: always wrap the original function
    return getattr(f, "_orig", f)
_orig_stage, _orig_loader, _orig_save = _unwrap(get_current_stage), _unwrap(build_loader), _unwrap(save_checkpoint)

def _mem():
    return torch.cuda.max_memory_allocated(), torch.cuda.max_memory_reserved()

def get_current_stage(step):                             # cell 13 calls this at the start of every step
    s = _orig_stage(step)
    if _stamps:                                          # store the peak of the step that just finished
        T_prev = _stamps[-1][1]
        a, r = _peak.get(T_prev, (0, 0)); ma, mr = _mem()
        _peak[T_prev] = (max(a, ma), max(r, mr))
    if not _stamps or _stamps[-1][1] != s.t:             # new stage → start its own peak
        torch.cuda.reset_peak_memory_stats()
    _stamps.append((time.time(), s.t))
    return s
get_current_stage._orig = _orig_stage

def build_loader(*a, **k):                               # the test loader marks the start of validation
    ld = _orig_loader(*a, **k)
    if k.get("split") == "test":
        _t["n_val_full"] = len(ld.dataset)
        _t["n_val_used"] = min(VAL_LIMIT, len(ld.dataset))
        ld = DataLoader(Subset(ld.dataset, range(_t["n_val_used"])), batch_size=1, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
        _t["val0"] = time.time()
        _t["peak_before_val"] = _mem()
    return ld
build_loader._orig = _orig_loader

def save_checkpoint(step, data_epoch, tag=""):           # the final save marks the end of validation
    if tag == "final":
        _t["val1"] = time.time()
        _t["peak_with_val"] = _mem()
    return _orig_save(step, data_epoch, tag)
save_checkpoint._orig = _orig_save

torch.cuda.reset_peak_memory_stats()
print("Debug setup ready on", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB)")

cell 13

In [ ]:
max_steps     = cfg.debug_steps if cfg.debug else cfg.max_steps
current_stage = get_current_stage(global_step)

# Validation: fixed centred window of the largest stage, so loss/val_rec is comparable across stages
VAL_N_TOK = max(s.t for s in cfg.t_schedule) // TUBELET_SIZE   # 4 tubelets (T=8); depends on cfg.t_schedule
VAL_TOK0  = (TEACHER_T // TUBELET_SIZE - VAL_N_TOK) // 2       # 2 — centred in the 8 teacher tubelets
VAL_F0    = VAL_TOK0 * TUBELET_SIZE                            # 4

loader      = build_loader(TEACHER_T, current_stage.batch_size, epoch=data_epoch)   # loader always yields 16 frames
loader_iter = iter(loader)

anyup.train()

pbar = tqdm(range(global_step, max_steps), initial=global_step, total=max_steps, desc="Training")

for step in pbar:

    new_stage = get_current_stage(step)
    if new_stage.t != current_stage.t or new_stage.batch_size != current_stage.batch_size:
        print(f"\n[Step {step}] Curriculum: T={current_stage.t}→{new_stage.t}, "
              f"BS={current_stage.batch_size}→{new_stage.batch_size}")
        current_stage = new_stage
        data_epoch   += 1
        loader        = build_loader(TEACHER_T, current_stage.batch_size, epoch=data_epoch)
        loader_iter   = iter(loader)

    try:
        batch = next(loader_iter)
    except StopIteration:
        data_epoch   += 1
        loader        = build_loader(TEACHER_T, current_stage.batch_size, epoch=data_epoch)
        loader_iter   = iter(loader)
        batch         = next(loader_iter)

    video = batch["video"].to(device)
    B, T, C, S, _ = video.shape

    with torch.no_grad():
        crop_high, crop_low = sample_tube_crop(video)                    # (B, 16, ...)
    f0, tok0, n_tok = sample_student_window(current_stage.t, TEACHER_T, TUBELET_SIZE)  # student T drives VRAM

    with autocast(device_type=device.type, dtype=torch.bfloat16):

        gt_full  = extract_teacher_features(encoder, crop_high, device)  # (B, 768, 8, 14, 14)
        gui_full = extract_teacher_features(encoder, crop_low,  device)
        crop_high_s, gt_feats       = take_window(crop_high, gt_full,  f0, tok0, n_tok, TUBELET_SIZE)  # (B, T, ...), (B, 768, T_tok, 14, 14)
        crop_low_s,  guidance_feats = take_window(crop_low,  gui_full, f0, tok0, n_tok, TUBELET_SIZE)
        T_tok = n_tok

        # Downsample to 7×7 so AnyUp has something to actually upsample
        B_, C_, Tt_, H_, W_ = guidance_feats.shape
        guidance_feats_low = F.interpolate(
            guidance_feats.reshape(B_ * Tt_, C_, H_, W_),
            size=(TOKEN_H_LOW, TOKEN_W_LOW),                   # 7×7
            mode="bilinear", align_corners=False,
        ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)       # (B, 768, T_tok, 7, 7)

        out_size = (TOKEN_H, TOKEN_W)

        # ── AnyUp3D forward ───────────────────────────────────────────────────
        crop_high_5d = crop_high_s.permute(0, 2, 1, 3, 4)        # (B, 3, T, 224, 224) — student window only
        pred_feats   = anyup(crop_high_5d, guidance_feats_low, output_size=out_size)
        # guidance_feats: (B, 768, T_tok, 7, 7) — coarse features to upsample
        # crop_high_5d:   (B, 3,   T,     224, 224) — HR pixels for spatial guidance
        # pred_feats:     (B, 768, T_tok, 14, 14) — upsampled output                          # ← FIX

        loss_rec = cosine_mse_3d(pred_feats, gt_feats) * cfg.lambda_cos_mse

        loss_inp = torch.tensor(0.0, device=device)
        if cfg.lambda_input_consistency > 0:
            loss_inp = input_consistency_loss(pred_feats, guidance_feats_low) \
                       * cfg.lambda_input_consistency

        loss_aug = torch.tensor(0.0, device=device)
        if cfg.lambda_self_consistency > 0:
            aug_low    = crop_low + 0.05 * torch.randn_like(crop_low)
            with torch.no_grad():
                gt_aug = extract_teacher_features(encoder, aug_low, device)[:, :, tok0:tok0 + n_tok]
            B_, C_, Tt_, H_, W_ = gt_aug.shape
            gt_aug_low = F.interpolate(
                gt_aug.reshape(B_ * Tt_, C_, H_, W_),
                size=(TOKEN_H_LOW, TOKEN_W_LOW),
                mode="bilinear", align_corners=False,
            ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)
            pred_aug = anyup(crop_high_5d, gt_aug_low, output_size=out_size)
            loss_aug   = cosine_mse_3d(pred_aug, pred_feats.detach()) \
                         * cfg.lambda_self_consistency

        loss_temp = torch.tensor(0.0, device=device)
        t_lam = get_temporal_lambda(step)
        if T_tok > 1 and t_lam > 0:
            feat_diff = pred_feats[:, :, 1:] - pred_feats[:, :, :-1]
            gate      = get_rgb_diff_gate(crop_low_s, T_tok)
            if gate.shape[-2:] != feat_diff.shape[-2:]:
                Tt   = T_tok - 1
                gate = F.interpolate(
                    gate.reshape(B * Tt, 1, *gate.shape[-2:]),
                    size=feat_diff.shape[-2:],
                    mode="bilinear", align_corners=False,
                ).reshape(B, 1, Tt, *feat_diff.shape[-2:])
            loss_temp = (feat_diff.pow(2) * gate).mean() * t_lam

        total_loss = loss_rec + loss_inp + loss_aug + loss_temp

    optimizer.zero_grad()
    scaler.scale(total_loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(anyup.parameters(), cfg.grad_clip_max_norm)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    if step % cfg.log_every_n_steps == 0:
        lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("loss/total",    total_loss.item(), step)
        writer.add_scalar("loss/rec",      loss_rec.item(),   step)
        writer.add_scalar("loss/inp",      loss_inp.item(),   step)
        writer.add_scalar("loss/aug",      loss_aug.item(),   step)
        writer.add_scalar("loss/temporal", loss_temp.item(),  step)
        writer.add_scalar("lr",            lr,                step)
        writer.add_scalar("curriculum/T",  current_stage.t,   step)
        writer.add_scalar("curriculum/bs", current_stage.batch_size, step)
        writer.add_scalar("data/epoch",    data_epoch,        step)
        pbar.set_postfix({
            "loss": f"{total_loss.item():.4f}",
            "rec":  f"{loss_rec.item():.4f}",
            "inp":  f"{loss_inp.item():.4f}",
            "T":    current_stage.t,
            "lr":   f"{lr:.2e}",
        })

    if (step + 1) % cfg.save_every_n_steps == 0:
        save_checkpoint(step + 1, data_epoch)

    if (step + 1) % cfg.val_every_n_steps == 0:
        anyup.eval()
        val_loader = build_loader(TEACHER_T, 1, split="test")   # batch 1: two 16-frame teacher passes per clip
        val_losses = []
        with torch.no_grad():
            for val_batch in val_loader:
                val_video = val_batch["video"].to(device)
                val_high, val_low = sample_tube_crop(val_video)
                with autocast(device_type=device.type, dtype=torch.bfloat16):
                    val_gt_full  = extract_teacher_features(encoder, val_high, device)
                    val_gui_full = extract_teacher_features(encoder, val_low,  device)
                    val_high_s, val_gt  = take_window(val_high, val_gt_full,  VAL_F0, VAL_TOK0, VAL_N_TOK, TUBELET_SIZE)
                    val_low_s,  val_gui = take_window(val_low,  val_gui_full, VAL_F0, VAL_TOK0, VAL_N_TOK, TUBELET_SIZE)
                    B_, C_, Tt_, H_, W_ = val_gui.shape
                    val_gui_low = F.interpolate(
                        val_gui.reshape(B_ * Tt_, C_, H_, W_),
                        size=(TOKEN_H_LOW, TOKEN_W_LOW),
                        mode="bilinear", align_corners=False,
                    ).reshape(B_, C_, Tt_, TOKEN_H_LOW, TOKEN_W_LOW)
                    val_pred = anyup(val_high_s.permute(0,2,1,3,4), val_gui_low, output_size=(TOKEN_H, TOKEN_W))
                    val_losses.append(cosine_mse_3d(val_pred, val_gt).item())
        writer.add_scalar("loss/val_rec", sum(val_losses) / len(val_losses), step + 1)
        anyup.train()

    if cfg.debug and step >= cfg.debug_steps - 1:
        print(f"Debug: stopping at step {step}.")
        break


save_checkpoint(max_steps, data_epoch, tag="final")
writer.flush(); writer.close()
print("Training complete.")

Cell 13.5

In [ ]:
# ── Cell 13.5 — TEMPORARY: print what cell 12.5 recorded during cell 13. ──
import statistics as st
G = 2**30
steps = _stamps[1:]                                      # drop the call made before the loop
T_of  = {i: T for i, (_, T) in enumerate(steps)}         # step number -> T

# Step durations: from one step's start to the next one's
dur = defaultdict(list)
for (t0, T), (t1, _) in zip(steps, steps[1:]):
    dur[T].append(t1 - t0)
if steps and "val0" in _t:                               # the last step ends where validation starts
    dur[steps[-1][1]].append(_t["val0"] - steps[-1][0])

# Peak memory of the last stage (its last step had no "next step" to record it)
if steps:
    T_last = steps[-1][1]
    a, r = _peak.get(T_last, (0, 0))
    pa, pr = _t.get("peak_before_val", _mem())
    _peak[T_last] = (max(a, pa), max(r, pr))

total = torch.cuda.get_device_properties(0).total_memory
print(f"GPU: {torch.cuda.get_device_name(0)}  ({total / G:.1f} GiB total)\n")

print("── Speed and memory per stage ──")
for T in sorted(dur):
    d = dur[T][1:] or dur[T]                             # skip each stage's first step (loader start-up)
    a, r = _peak.get(T, (0, 0))
    print(f"T={T}: {len(dur[T])} steps | median {st.median(d):.2f} s/step ({1 / st.median(d):.2f} it/s) | "
          f"first step {dur[T][0]:.1f} s | peak {a / G:.2f} GiB allocated, {r / G:.2f} GiB reserved "
          f"({100 * r / total:.0f}% of GPU)")

print("\n── Validation ──")
if "val0" in _t and "val1" in _t:
    sec = _t["val1"] - _t["val0"]
    per_clip = sec / _t["n_val_used"]
    a, r = _t["peak_with_val"]
    print(f"{_t['n_val_used']} of {_t['n_val_full']} clips in {sec:.0f} s ({per_clip:.2f} s/clip) "
          f"→ full pass ≈ {per_clip * _t['n_val_full'] / 60:.1f} min")
    print(f"peak incl. validation: {a / G:.2f} GiB allocated, {r / G:.2f} GiB reserved")
else:
    print("validation did not run (cell 13 stopped early?)")

print("\n── Losses per step ──")
tags = ["loss/total", "loss/rec", "loss/inp", "loss/aug", "loss/temporal"]
vals = {t: dict(_scalars.get(t, [])) for t in tags}
missing = [t for t in tags + ["loss/val_rec"] if t not in _scalars]
if missing:
    print("not logged:", missing)
print("step  T     total      rec      inp      aug  temporal")
for s in sorted(vals["loss/total"]):
    print(f"{s:4d} {T_of.get(s, '?'):>2} " + " ".join(f"{vals[t].get(s, float('nan')):8.4f}" for t in tags))
val = _scalars.get("loss/val_rec", [])
if val:
    print("val_rec:", " ".join(f"step {s}: {x:.4f}" for s, x in val))
everything = [x for t in tags for _, x in _scalars.get(t, [])] + [x for _, x in val]
print(f"\nall losses finite: {all(math.isfinite(x) for x in everything)}  ({len(everything)} values)")
print("(loss/temporal = 0 is expected: its weight ramps up from 0 over 5000 steps)")

cell 14

In [ ]:
anyup.eval()
with torch.no_grad():
    dummy_video = torch.randn(1, 4, 3, cfg.source_size, cfg.source_size, device=device)
    dummy_high, dummy_low = sample_tube_crop(dummy_video)

    dummy_gt  = torch.randn(1, ENCODER_DIM, 2, TOKEN_H, TOKEN_W, device=device)
    dummy_gui = torch.randn(1, ENCODER_DIM, 2, TOKEN_H, TOKEN_W, device=device)

    dummy_low_5d  = dummy_low.permute(0, 2, 1, 3, 4)
    dummy_high_5d = dummy_high.permute(0, 2, 1, 3, 4)
    out = anyup(dummy_low_5d, dummy_high_5d, output_size=(TOKEN_H, TOKEN_W))

    assert out.shape == dummy_gt.shape, f"Shape mismatch: {out.shape} vs {dummy_gt.shape}"

    gate = get_rgb_diff_gate(dummy_low, T_tok=2)
    assert gate.shape == (1, 1, 1, TOKEN_H, TOKEN_W), f"Gate shape wrong: {gate.shape}"

    loss_inp_test = input_consistency_loss(out, dummy_gui)
    assert loss_inp_test.ndim == 0, "input_consistency_loss should return a scalar"

    print(f"source → crop_high : {dummy_high.shape}")
    print(f"source → crop_low  : {dummy_low.shape}")
    print(f"AnyUp  input       : {dummy_low_5d.shape}")
    print(f"AnyUp  output      : {out.shape}")
    print(f"RGB-diff gate      : {gate.shape}")
    print(f"L_inp (scalar)     : {loss_inp_test.item():.4f}")
    print("Smoke test passed ✓")

anyup.train()